#Simple Linear Regression (SLR) vs Multiple Linear Regression (MLR)

**Student:** Paul Boison  
**Student ID:** PUIT-23110017  
**Dataset:** `regression_data.csv` (5,000 synthetic observations)  
**Objective:** Train and compare a Simple Linear Regression model and a Multiple Linear Regression model.

## Research question
Does using several relevant features improve prediction of `target_score` compared with using only `experience_years`?

## Dataset features
- `experience_years`: years of relevant experience
- `education_years`: years of education
- `weekly_study_hours`: typical study hours per week
- `projects_completed`: number of completed projects
- `internship_months`: months of internship experience
- `target_score`: continuous outcome to predict

This is a synthetic dataset created for learning and demonstration; it does not represent real people or official records.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Load and inspect the dataset

In [ ]:
df = pd.read_csv("regression_data.csv")

print("Dataset shape:", df.shape)
display(df.head())
display(df.describe())
print("\nMissing values per column:")
display(df.isnull().sum())

In [ ]:
# Confirm the dataset has 5,000 rows and no missing values.
assert len(df) == 5000, "Expected exactly 5,000 rows."
assert df.isnull().sum().sum() == 0, "Dataset contains missing values."
print("Validation passed: 5,000 rows and no missing values.")

## 3. Exploratory data analysis

The target variable is continuous, so regression is appropriate. The following plots show the target distribution and the relationship between experience and the target.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df["target_score"], kde=True, ax=axes[0])
axes[0].set_title("Distribution of target_score")
axes[0].set_xlabel("Target score")

sns.scatterplot(data=df, x="experience_years", y="target_score", alpha=0.35, ax=axes[1])
sns.regplot(data=df, x="experience_years", y="target_score", scatter=False, color="red", ax=axes[1])
axes[1].set_title("Experience years vs target score")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

## 4. Prepare training and testing data

Both models use the same 80/20 train-test split to make the comparison fair. The same random seed is used for reproducibility.

In [ ]:
target = "target_score"
X_slr = df[["experience_years"]]  # One predictor: Simple Linear Regression
X_mlr = df[[
    "experience_years",
    "education_years",
    "weekly_study_hours",
    "projects_completed",
    "internship_months"
]]  # Multiple predictors: Multiple Linear Regression
y = df[target]

X_slr_train, X_slr_test, y_train, y_test = train_test_split(
    X_slr, y, test_size=0.20, random_state=RANDOM_STATE
)
X_mlr_train, X_mlr_test, _, _ = train_test_split(
    X_mlr, y, test_size=0.20, random_state=RANDOM_STATE
)

print("Training observations:", len(y_train))
print("Testing observations:", len(y_test))

## 5. Train the Simple Linear Regression model

In [ ]:
slr_model = LinearRegression()
slr_model.fit(X_slr_train, y_train)
slr_predictions = slr_model.predict(X_slr_test)

print("SLR intercept:", round(slr_model.intercept_, 3))
print("SLR coefficient for experience_years:", round(slr_model.coef_[0], 3))

## 6. Train the Multiple Linear Regression model

In [ ]:
mlr_model = LinearRegression()
mlr_model.fit(X_mlr_train, y_train)
mlr_predictions = mlr_model.predict(X_mlr_test)

coefficients = pd.DataFrame({
    "Feature": X_mlr.columns,
    "Coefficient": mlr_model.coef_
})
display(coefficients)
print("MLR intercept:", round(mlr_model.intercept_, 3))

## 7. Evaluate and compare both models

- **MAE:** average absolute prediction error; lower is better.
- **MSE:** average squared prediction error; lower is better.
- **RMSE:** square root of MSE, in the target's units; lower is better.
- **R²:** proportion of target variation explained by the model; closer to 1 is generally better.

In [ ]:
def evaluate_model(model_name, y_true, y_pred):
    return {
        "Model": model_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

results = pd.DataFrame([
    evaluate_model("Simple Linear Regression", y_test, slr_predictions),
    evaluate_model("Multiple Linear Regression", y_test, mlr_predictions)
]).set_index("Model")

display(results.round(3))

In [ ]:
metrics = ["MAE", "MSE", "RMSE", "R2"]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, metric in zip(axes, metrics):
    results[metric].plot(kind="bar", ax=ax, title=metric)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=35)
plt.suptitle("SLR vs MLR performance comparison")
plt.tight_layout()
plt.show()

## 8. Actual vs predicted values

In [ ]:
comparison = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "SLR predicted": slr_predictions,
    "MLR predicted": mlr_predictions
}).reset_index(drop=True)

display(comparison.head(10))

plt.figure(figsize=(6, 6))
plt.scatter(y_test, slr_predictions, alpha=0.45, label="SLR")
plt.scatter(y_test, mlr_predictions, alpha=0.45, label="MLR")
min_val = min(y_test.min(), slr_predictions.min(), mlr_predictions.min())
max_val = max(y_test.max(), slr_predictions.max(), mlr_predictions.max())
plt.plot([min_val, max_val], [min_val, max_val], "k--", label="Perfect prediction")
plt.xlabel("Actual target score")
plt.ylabel("Predicted target score")
plt.title("Actual vs predicted values")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Residual analysis

In [ ]:
# Residual = actual value - predicted value
slr_residuals = y_test.to_numpy() - slr_predictions
mlr_residuals = y_test.to_numpy() - mlr_predictions

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(slr_predictions, slr_residuals, alpha=0.45)
axes[0].axhline(0, linestyle="--", color="red")
axes[0].set_title("SLR residuals")
axes[0].set_xlabel("Predicted values")
axes[0].set_ylabel("Residual (actual - predicted)")

axes[1].scatter(mlr_predictions, mlr_residuals, alpha=0.45)
axes[1].axhline(0, linestyle="--", color="red")
axes[1].set_title("MLR residuals")
axes[1].set_xlabel("Predicted values")
axes[1].set_ylabel("Residual (actual - predicted)")
plt.tight_layout()
plt.show()

### Ten test observations: actual values, predictions, and residuals

In [ ]:
residual_sample = pd.DataFrame({
    "Actual": y_test.to_numpy()[:10],
    "SLR predicted": slr_predictions[:10],
    "SLR residual": slr_residuals[:10],
    "MLR predicted": mlr_predictions[:10],
    "MLR residual": mlr_residuals[:10]
})
display(residual_sample.round(2))

## 10. Conclusion

In [ ]:
best_by_rmse = results["RMSE"].idxmin()
best_by_r2 = results["R2"].idxmax()

print(f"Model with the lower RMSE: {best_by_rmse}")
print(f"Model with the higher R²: {best_by_r2}")

slr_rmse = results.loc["Simple Linear Regression", "RMSE"]
mlr_rmse = results.loc["Multiple Linear Regression", "RMSE"]
slr_r2 = results.loc["Simple Linear Regression", "R2"]
mlr_r2 = results.loc["Multiple Linear Regression", "R2"]

print("\nInterpretation:")
if mlr_rmse < slr_rmse and mlr_r2 > slr_r2:
    print("MLR performed better on this test set: it had lower RMSE and higher R².")
elif slr_rmse < mlr_rmse and slr_r2 > mlr_r2:
    print("SLR performed better on this test set according to RMSE and R².")
else:
    print("The metrics do not all point to the same model. Consider the trade-offs between prediction error and explained variation.")

print("SLR uses only experience_years, while MLR uses five predictors.")
print("The comparison suggests whether adding relevant features improves prediction for this synthetic dataset.")

## 11. Key findings and limitations

1. Both models were trained and evaluated on the same 80/20 split.
2. SLR uses only `experience_years`; MLR uses five input features.
3. MAE, MSE, RMSE, and R² provide complementary measures of predictive performance.
4. Residuals are calculated as actual minus predicted values. Residuals near zero indicate predictions close to the actual values.
5. This dataset is synthetic and was constructed with several features contributing to the target. Results should not be generalized to real-world data.

**Submission files:** `regression_comparison.ipynb`, `regression_data.csv`, `requirements.txt`, and `.gitignore`.